In [1]:
# 1. Initialization

import matplotlib.pyplot as plt
from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
    TensorDataset,
    random_split
)
import os
import pandas as pd
import numpy as np
from pathlib import Path
from torchvision import datasets
from torchvision.transforms import v2
import torch
from torch import nn

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    else:
        return torch.device("cpu")

device = get_device()

print('torch:', torch.__version__)
print('built CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print(f"Selected device: {device}")

torch: 2.13.0+cu130
built CUDA: 13.0
CUDA available: True
device: NVIDIA GeForce RTX 5060 Laptop GPU
Selected device: cuda:0


# Data storage for this exercies

- Data storage: CPU
- DataLoader batches: CPU
- Training loop data: GPU (Transfer batched from CPU)
- Model parameters: GPU
- Loss computation: GPU

Storing everything on GPU is generally not a good idea cuz VRAM expensive D:. Generally we only want to store data actively participating in computation in GPU.

# Dataset vs DataLoader
## Dataset
- The raw data storage, containing samples and labels. Can be random-accessed and has some functions for manipulation.
## DataLoader
- An iterable wrapper for Dataset for easy data access.
- Has some useful features like sampling, shuffling, etc...
- Creating for ease of loading data.

## Creating custom dataset

Custom Dataset class must implement three functions
- __init__: Instantiate object (classic python)
- __len__: Getting length of object (classic python)
- __getitem__: Getting item at a given index (classic python)

Really doesn't matter how you get there (in-core, out-of-core,...) as long as you can provide these funcs.

In [2]:
class TestClassificationDataset(Dataset):
    def __init__(self) -> None:
        # Hardcoding dataset
        self.features = torch.tensor(
            [
                [1.0, 2.0],
                [3.0, 4.0],
                [5.0, 6.0],
                [7.0, 8.0]
            ],
            dtype=torch.float32,
        )

        self.targets = torch.tensor(
            [0, 1, 0, 1],
            dtype=torch.long
        )
    
    def __len__(self) -> int:
        return len(self.targets)
    
    def __getitem__(
        self,
        index: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        return (
            self.features[index],
            self.targets[index],
        )

In [3]:
test_dataset = TestClassificationDataset()

sample_features, sample_target = test_dataset[2]

print(len(test_dataset))
print(sample_features.shape)
print(sample_target.shape)

4
torch.Size([2])
torch.Size([])


In [4]:
# Test DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size = 3,
    shuffle = False
)

test_loader_dl = DataLoader(
    test_dataset,
    batch_size = 3,
    shuffle = False,
    drop_last = True,
)

test_loader_dl2 = DataLoader(
    test_dataset,
    batch_size = 2,
    shuffle = False,
    drop_last = True
)

for batch_index, (batch_features, batch_targets) in enumerate(test_loader):
    print(
        batch_index, batch_features.shape, batch_targets.shape,
    )
for batch_index, (batch_features, batch_targets) in enumerate(test_loader_dl):
    print(
        batch_index, batch_features.shape, batch_targets.shape,
    )
for batch_index, (batch_features, batch_targets) in enumerate(test_loader_dl2):
    print(
        batch_index, batch_features.shape, batch_targets.shape,
    )

0 torch.Size([3, 2]) torch.Size([3])
1 torch.Size([1, 2]) torch.Size([1])
0 torch.Size([3, 2]) torch.Size([3])
0 torch.Size([2, 2]) torch.Size([2])
1 torch.Size([2, 2]) torch.Size([2])


In [5]:
# RNG yes i love gambling
generator = torch.Generator().manual_seed(42)

shuffled_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=True,
    generator=generator,
)

shuffled_features, _ = next(
    iter(shuffled_loader)
)
print(shuffled_features)

tensor([[1., 2.],
        [5., 6.],
        [3., 4.],
        [7., 8.]])


In [6]:
# Custom CSV dataset

data_dir = (Path(os.getcwd()) / ".." / "data" / "06_synthetic_binary").resolve()
print(data_dir)
data_dir.mkdir(parents=True, exist_ok=True)

csv_path = data_dir / "synthetic_binary.csv"

/home/halzyon/Data/coding_work/projects/pytorch-lab/data/06_synthetic_binary


In [7]:
torch.manual_seed(17)

num_examples = 2000
num_features = 6

features = torch.randn(
    num_examples,
    num_features,
)

# Hardcode true weights
true_weights = torch.tensor(
    [1.5, -2.0, 0.75, 0.0, 1.0, -0.5]
)

noise = 0.05 * torch.randn(num_examples)

scores = features @ true_weights + noise

# Label rule: 1 * (x_{i}^T * w + epsilon_{i} > 0)
# That is, 1 if positive score, 0 otherwise.
labels = (scores > 0).to(torch.int64)

feature_columns = [
    f"feature_{index}"
    for index in range(num_features)
]

df = pd.DataFrame(
    features.numpy(),
    columns = feature_columns,
)

df["label"] = labels.numpy()

df.to_csv(
    csv_path,
    index=False,
)

assert csv_path.exists()
print(df.shape)
print(df.columns)

(2000, 7)
Index(['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
       'feature_5', 'label'],
      dtype='str')


In [ ]:
# Our custom Dataset (in-memory)

class CSVDataset(Dataset):
    def __init__(
        self,
        csv_path: str | Path,
        feature_columns: list[str],
        label_column: str,
    ) -> None:
        self.csv_path = Path(csv_path)
        self.feature_columns = feature_columns
        self.label_column = label_column
        
        self.df = pd.read_csv(self.csv_path)

        required_columns = set(self.feature_columns) | set([self.label_column])

        missing_columns = required_columns - set(df.columns)

        if missing_columns:
            raise ValueError(
                f"Missing columns: {missing_columns}"
            )
        
        self.features = torch.tensor(
            df[self.feature_columns].to_numpy(),
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            df[self.label_column].to_numpy(),
            dtype=torch.long
        )

    def __len__(self) -> int:
        return self.labels.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return (self.features[idx], self.labels[idx])

In [ ]:
print(type(csv_path))

csv_dataset = CSVDataset(
    csv_path = csv_path,
    feature_columns = feature_columns,
    label_column="label"
)

sample_features, sample_label = csv_dataset[0]

print(sample_features.shape)
print(sample_features.dtype)
print(sample_label.shape)
print(sample_label.dtype)

<class 'pathlib.PosixPath'>
torch.Size([6])
torch.float32
torch.Size([])
torch.int64


In [ ]:
# Validate against original .csv

preview = pd.read_csv(csv_path)

expected_features = torch.tensor(
    preview.loc[0, feature_columns].to_numpy(dtype="float32")
)

expected_label = torch.tensor(
    preview.loc[0, "label"]
)

assert torch.allclose(
    expected_features, sample_features
)

assert sample_target.item() == expected_label.item()

In [ ]:
# Loader for dataset
csv_loader = DataLoader(
    csv_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

batch_features, batch_targets = next(iter(csv_loader))

print(batch_features.shape)
print(batch_targets.shape)

torch.Size([64, 6])
torch.Size([64])


## Reviewing basic concepts:
- Training set: Data used to fit model parameters
- Validation set: This for validating model during training phase, useful for
    - Determining hyper-params
    - Comparing different algorithms
    - Overfitting signal
- Test set: Stay hidden until final evaluation.

A typical data split would be:
- 70\% train
- 15\% vaidation
- 15\% test

In [ ]:
# Creating Subsets for train, val, test

split_generator = (torch.Generator().manual_seed(67))

train_subset, val_subset, test_subset = (
    random_split(
        csv_dataset,
        lengths=[0.7, 0.15, 0.15],
        generator=split_generator
    )
)

print(len(train_subset))
print(len(val_subset))
print(len(test_subset))

1400
300
300


In [36]:
train_indicies = set(train_subset.indices)
val_indicies = set(val_subset.indices)
test_indicies = set(test_subset.indices)

assert train_indicies.isdisjoint(val_indicies)
assert train_indicies.isdisjoint(test_indicies)
assert val_indicies.isdisjoint(test_indicies)

In [ ]:
# Normalizing data training data (mean = 0, variance = 1)

train_index_tensor = torch.tensor(
    train_subset.indices,
    dtype=torch.long
)

train_feature_raw = (
    csv_dataset.features[
        train_index_tensor
    ]
)

train_mean = train_feature_raw.mean(dim=0)
train_std = train_feature_raw.std(dim=0)

# Replacing zero_variance features with 1
train_std = torch.where(
    train_std > 0,
    train_std,
    torch.ones_like(train_std),
)

print(train_mean.shape, train_std.shape)
assert(torch.all(train_std > 0))

torch.Size([6]) torch.Size([6])


In [41]:
# Standardizer
def standardizer(
    features: torch.Tensor,
) -> torch.Tensor:
    mean_val = features.mean(dim=0)
    std_val = features.std(dim=0)

    # Replacing zero_variance features with 1
    std_val = torch.where(
        std_val > 0,
        std_val,
        torch.ones_like(std_val),
    )

    return (features - mean_val) / std_val

In [53]:
# Function to create normalized features

def subset_to_tensor_dataset(
    subset: Subset
) -> TensorDataset:
    indices = torch.tensor(
        subset.indices,
        dtype=torch.long,
    ).clone()

    raw_features = (
        csv_dataset.features[
            indices
        ]
    )

    normalized_features = standardizer(raw_features)

    return TensorDataset(normalized_features)

In [54]:
train_dataset = subset_to_tensor_dataset(train_subset)
val_dataset = subset_to_tensor_dataset(val_subset)
test_dataset = subset_to_tensor_dataset(test_subset)

In [58]:
# Validating normalization

normalized_train_features = train_dataset.tensors[0]

assert torch.allclose(
    normalized_train_features.mean(dim=0),
    torch.zeros(num_features),
    atol=1e-5,
)

assert torch.allclose(
    normalized_train_features.std(dim=0),
    torch.ones(num_features),
    atol=1e-5,
)

In [59]:
# Loader for normalized data
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)